In [1]:
from PIL import Image
from transformers import AlignProcessor, AlignModel
import torch
from torch.nn.functional import cosine_similarity
import requests
from io import BytesIO

In [2]:
processor = AlignProcessor.from_pretrained("kakaobrain/align-base")
model = AlignModel.from_pretrained("kakaobrain/align-base")

preprocessor_config.json:   0%|          | 0.00/508 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/5.25k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/399 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  690MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  690MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1394 [00:00<?, ?it/s]

In [3]:
texts = ["A cat sitting on a mat", "A dog playing with a ball"]
image_urls = [
    "https://i.postimg.cc/7hJL9KK4/fluffy-siberian-cat-sitting-on-600nw-2150187551.png",
    "https://i.postimg.cc/4xgzKc3y/images.jpg"
]

In [4]:
images = []
for url in image_urls:
    resp = requests.get(url)
    if resp.status_code == 200:
        img = Image.open(BytesIO(resp.content)).convert("RGB")
        images.append(img)

In [5]:
inputs = processor(
    images=images, text=texts, return_tensors="pt",
    padding='max_length', max_length=128, truncation=True
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

In [6]:
outputs = model(**inputs)

image_embeds = outputs.image_embeds
text_embeds = outputs.text_embeds

In [7]:
similarity = cosine_similarity(image_embeds, text_embeds)

for i, score in enumerate(similarity):
    print(f"Similarity score for pair {i+1}: {score.item():.4f}")

Similarity score for pair 1: 0.2409
Similarity score for pair 2: 0.2758
